# PPO Rollout Mechanics

## Overview

Before PPO can update anything, it first needs **experience**.

The **rollout** is simply:

> Run the current policy in the environment and record what happens.

---

## Step 1: Start with the Current Actor + Critic

At timestep $t$:

```
State: S_t
   │
   ├── Actor → action probabilities
   │
   └── Critic → V(S_t)
```

**Actor outputs** (probabilities):

```
LEFT  = 0.2
RIGHT = 0.7
UP    = 0.1
```

Suppose it **samples**: `action = RIGHT`

We also **record**:

```python
old_log_prob = log(0.7)
```

And the **critic gives**:

```python
V(S_t) = 5.0
```

## Step 2: Execute the Action

Environment responds:

```
S_t
 ↓ RIGHT
reward = +2
 ↓
S_{t+1}
```

We **record**:

```python
state        = S_t
action       = RIGHT
reward       = +2
old_log_prob = log(0.7)
value        = 5.0
```

Then **repeat** from $S_{t+1}$.

## Step 3: What Does the Rollout Buffer Contain?

For a trajectory:

```
S0 → A0 → R0 → S1
S1 → A1 → R1 → S2
S2 → A2 → R2 → S3
...
```

We store roughly:

```
states
actions
rewards
dones
old_log_probs
values
```

**Key Point:** We collect this using the **current policy before updating it**.

## Step 4: Why Store old_log_prob?

This is directly connected to **PPO clipping**.

Suppose during rollout:

$$\pi_{\text{old}}(\text{RIGHT} \mid S) = 0.70$$

We **save**: $\log(0.70)$

Then we **train the actor**.

After an update, the actor might now give:

$$\pi_{\text{new}}(\text{RIGHT} \mid S) = 0.80$$

We **compare**:

$$r_t = \frac{\text{new probability}}{\text{old probability}} = \frac{0.80}{0.70} = 1.14$$

**That is PPO's ratio.**

**Without the old probability**, we don't know how much the policy changed relative to the policy that generated the data.

## Step 5: Why Store the Critic's Value?

We need it to **calculate the advantage later**.

Suppose:

```
V(S_t)      = 5
reward      = 2
V(S_{t+1})  = 6
```

Then the **TD error** (temporal difference) is:

$$\delta = 2 + \gamma \times 6 - 5$$

With $\gamma = 0.9$:

$$\delta = 2 + 5.4 - 5 = 2.4$$

**Interpretation:** The outcome was **better than the critic expected**.

**GAE** then combines these TD errors across the trajectory to produce the final advantages.

## Step 6: The Rollout Phase Does NOT Update the Network

This distinction is **important**.

**During rollout:**

```
Actor
  ↓
action
  ↓
Environment
  ↓
reward
  ↓
store everything

(No gradient update yet.)
```

**After we've collected enough experience:**

```
rollout
   ↓
GAE
   ↓
advantages
   ↓
PPO optimization
```

## The Full Picture

```
        CURRENT POLICY
              │
              ▼
        ┌───────────┐
        │ Environment│
        └─────┬─────┘
              │
       ┌──────┴──────┐
       ▼             ▼
    reward          next state
       │
       └──────┬──────┘
              ▼
        Rollout Buffer
              │
       ┌──────┴───────┐
       ▼              ▼
    old log-prob     value
       │              │
       └──────┬───────┘
              ▼
             GAE
              │
              ▼
         advantages
              │
              ▼
        PPO optimization
```

## One Key Mental Model

| Phase | Purpose |
|---|---|
| **Rollout** | Collect evidence |
| **GAE** | Analyze that evidence |
| **PPO Update** | Change the actor/critic based on that analysis |

## Example: GAE Propagating Information Backward

Suppose instead:

```
V(S1) = 2
V(S2) = 2
V(S3) = 2
```

But the actual episode still produces:

```
+1, 0, +5
```

### Compute TD Errors at Each Step

**S3 (Terminal Step):**

$$\delta_3 = 5 - 2 = 3$$

**S2:**

$$\delta_2 = 0 + 0.9(2) - 2 = -0.2$$

**S1:**

$$\delta_1 = 1 + 0.9(2) - 2 = 0.8$$

### GAE Propagates Backward

Now **GAE propagates the later information backward**:

$$A_3 = 3$$

$$A_2 = -0.2 + (0.9 \times 0.95)(3) \approx 2.365$$

$$A_1 = 0.8 + (0.9 \times 0.95)(2.365) \approx 2.823$$

### Final Advantages

```
S1 → Advantage ≈ +2.82
S2 → Advantage ≈ +2.37
S3 → Advantage = +3
```

### The Purpose of GAE

**Key Insight:** The later reward **contributes to the advantage of earlier actions**.

This is the connection to value propagation:

```
environment reward
        ↓
TD errors
        ↓
GAE
        ↓
advantages
        ↓
PPO actor update
```

# PPO Actor Update

Now we connect all the pieces.

We have:

- **old policy** → generated the rollout
- **current policy** → we're training
- **advantage** → tells us whether the chosen action was good/bad
- **ratio** → tells us how much the current policy changed

---

## 1. One Timestep

Suppose during rollout:

```
state = S
chosen action = RIGHT
```

**Old policy:**

$$P(\text{RIGHT}|S) = 0.50$$

**Advantage:** $+2$ (positive → action was better than expected)

**During training**, the new policy currently says:

$$P_{\text{new}}(\text{RIGHT}|S) = 0.60$$

**Ratio:**

$$r_t = \frac{0.60}{0.50} = 1.2$$

**Basic objective:**

$$r_t \times A = 1.2 \times 2 = 2.4$$

**Interpretation:** We're rewarding the policy for **increasing a good action**.

## 2. What If It Increases Too Much?

Suppose the new policy becomes:

$$P_{\text{new}}(\text{RIGHT}|S) = 0.80$$

**Ratio:**

$$r_t = \frac{0.80}{0.50} = 1.6$$

**With PPO's** $\epsilon = 0.2$, **the allowed upper ratio is:** $1.2$

PPO compares:

| Type | Value |
|---|---|
| Normal objective | $1.6 \times 2 = 3.2$ |
| Clipped objective | $1.2 \times 2 = 2.4$ |

PPO takes the **minimum**:

$$\min(3.2, 2.4) = 2.4$$

**Result:** The optimizer gets **no additional reward** for pushing RIGHT from 0.8 even further.

## 3. What About a Bad Action?

Suppose:

```
advantage = -2
old probability = 0.50
new probability = 0.40
```

**Ratio:**

$$r_t = 0.8$$

**Normal objective:**

$$0.8 \times (-2) = -1.6$$

The policy **reduced the probability of a bad action**, which is what we want. PPO allows this.

But if the policy aggressively drops it:

```
new probability = 0.10
ratio = 0.2
```

PPO clips the ratio to 0.8, preventing the objective from benefiting from an **excessively large change**.

## 4. What Actually Gets Updated?

The **actor's neural-network weights**.

**The process:**

```
old log-prob
      +
new log-prob
      ↓
probability ratio
      +
advantage
      ↓
clipped PPO objective
      ↓
actor loss
      ↓
backpropagation
      ↓
actor weights
```

**The critic is updated separately** using its value loss:

```
target value
     ↓
critic prediction
     ↓
value loss
     ↓
critic weights
```

### Two Learning Problems

| Component | Task | Objective |
|---|---|---|
| **ACTOR** | "What action should I take?" | PPO clipped objective |
| **CRITIC** | "How valuable is this state?" | Value regression |

## 5. One Important Detail: Multiple Epochs

This is one of PPO's **big practical advantages**.

We **collect a rollout once**:

```
S1 → S2 → S3 → ... → Sn
```

Then calculate:

```
advantages
old log-probs
returns
```

And **reuse that same rollout** for several optimization epochs.

**Example:**

```
Rollout
  ↓
Epoch 1 → update
  ↓
Epoch 2 → update
  ↓
Epoch 3 → update
  ↓
Epoch 4 → update
```

**Key:** The `old_log_prob` **stays fixed** across all epochs.

The `new log-prob` **keeps changing** as the actor learns.

**That's exactly why the ratio can detect:**

> "We've now moved quite far from the policy that generated this data."

Once we've finished those epochs, we **throw the rollout away** and collect fresh experience with the new policy.

## The Complete PPO Cycle

```
1. Current actor interacts with environment
                ↓
2. Store states/actions/rewards/old log-probs/values
                ↓
3. Calculate returns + GAE advantages
                ↓
4. Freeze old log-probs
                ↓
5. Train actor + critic for several epochs
                ↓
6. Discard old rollout
                ↓
7. Collect fresh rollout
                ↓
8. Repeat
```

**That's the core PPO algorithm.**

## Student Question

> **Q:** For one episode or rollout, are they the same? Do we run multiple epochs?

## Answer: Episodes, Rollouts, and Epochs

Think of it like this:

| Term | Definition |
|---|---|
| **Episode** | One complete run until termination |
| **Rollout** | The experience we collect before an update. It can be one episode or a chunk of multiple episodes |
| **Epoch** | How many times we train on that already-collected rollout |

### Typical PPO Flow

```
Collect rollout
    ↓
Freeze old log-probs
    ↓
Calculate advantages
    ↓
Epoch 1 → update actor + critic
Epoch 2 → update actor + critic
Epoch 3 → update actor + critic
...
Epoch N
    ↓
Throw rollout away
    ↓
Collect fresh rollout
```

### Key Points

**Yes, multiple epochs happen on the same collected experience.**

**Importantly:**

- The **old probabilities stay fixed** across all those epochs
- The **current policy probabilities keep changing** as you optimize